In [ ]:
/home/benskim915/api/data_.pq

In [2]:
import pandas as pd
import glob

files = glob.glob('../parquet/MEDICINE_2025/MEDICINE_*.parquet')
print(files)
df = pd.read_parquet(files[0])

# 2. 첫 번째 행(index 0)의 값을 새로운 컬럼명으로 설정
df.columns = df.iloc[0]

# 3. 컬럼명으로 올라간 첫 번째 행 데이터 제거 및 인덱스 재정리
df = df.iloc[1:].reset_index(drop=True)

# 4. 컬럼명 이름(name) 속성 제거 (선택 사항)
df.columns.name = None

df.head()

['../parquet/MEDICINE_2025/MEDICINE_202508 (2026.09.30.).parquet', '../parquet/MEDICINE_2025/MEDICINE_202510 (2026.09.30.).parquet', '../parquet/MEDICINE_2025/MEDICINE_202505 (2026.09.30.).parquet', '../parquet/MEDICINE_2025/MEDICINE_202501 (2026.09.30.).parquet', '../parquet/MEDICINE_2025/MEDICINE_202511 (2026.09.30.).parquet', '../parquet/MEDICINE_2025/MEDICINE_202512 (2026.09.30.).parquet', '../parquet/MEDICINE_2025/MEDICINE_202509 (2026.09.30.).parquet', '../parquet/MEDICINE_2025/MEDICINE_202507 (2026.09.30.).parquet', '../parquet/MEDICINE_2025/MEDICINE_202503 (2026.09.30.).parquet', '../parquet/MEDICINE_2025/MEDICINE_202502 (2026.09.30.).parquet', '../parquet/MEDICINE_2025/MEDICINE_202506 (2026.09.30.).parquet', '../parquet/MEDICINE_2025/MEDICINE_202504 (2026.09.30.).parquet']


,STD_YYYYMM,WK_COMPN_4,AGE_G,SEX_TYPE,SIDO,YOYANG_CLSFC_CD_ADJ,OUT_IN_HOS,PRSCRPTN_TNDN_CNT,PRSCRPTN_AMT,PATIENT_CNT,PRSCRPTN_TOT_USE
0,202508,1007,0,1,11,1,OUT,2,4974,2,24.50
1,202508,1007,0,1,11,3,OUT,479,401155,347,2055.88
2,202508,1007,0,1,26,1,OUT,1,2110,1,10.00
3,202508,1007,0,1,26,3,OUT,47,40542,36,206.15
4,202508,1007,0,1,27,3,OUT,78,60923,54,333.74


In [3]:
df.dtypes

STD_YYYYMM             str
WK_COMPN_4             str
AGE_G                  str
SEX_TYPE               str
SIDO                   str
YOYANG_CLSFC_CD_ADJ    str
OUT_IN_HOS             str
PRSCRPTN_TNDN_CNT      str
PRSCRPTN_AMT           str
PATIENT_CNT            str
PRSCRPTN_TOT_USE       str
dtype: object

In [4]:
# 2. 수치형 컬럼 자료형 변환 (str -> float/int)
num_cols = ['PRSCRPTN_TNDN_CNT', 'PRSCRPTN_AMT', 'PATIENT_CNT', 'PRSCRPTN_TOT_USE']

for col in num_cols:
    # pd.to_numeric을 사용해 문자열을 숫자로 변환 (변환 불가 값은 NaN 처리 후 0으로 채움)
    df[col] = pd.to_numeric(df[col], errors='coerce')#.fillna(0)

In [6]:


# 3. 성분(WK_COMPN_4), 연령(AGE_G), 성별(SEX_TYPE) 기준 그룹화 및 집계
# 처방건수(PRSCRPTN_TNDN_CNT), 처방금액(PRSCRPTN_AMT), 환자수(PATIENT_CNT), 총사용량(PRSCRPTN_TOT_USE) 합산
summary_df = df.groupby(['STD_YYYYMM','WK_COMPN_4', 'AGE_G', 'SEX_TYPE'], as_index=False).agg({
    'PATIENT_CNT': 'sum',
    'PRSCRPTN_TNDN_CNT': 'sum',
    'PRSCRPTN_AMT': 'sum',
    'PRSCRPTN_TOT_USE': 'sum'
})

# 4. 결과 확인
summary_df.head()

,STD_YYYYMM,WK_COMPN_4,AGE_G,SEX_TYPE,PATIENT_CNT,PRSCRPTN_TNDN_CNT,PRSCRPTN_AMT,PRSCRPTN_TOT_USE
0,202508,1007,0,1,3525,5026,4392905,22931.95
1,202508,1007,0,2,2915,4066,3480797,18098.23
2,202508,1007,10,1,4040,4821,8718695,45751.95
3,202508,1007,10,2,3180,3739,6202937,32451.80
4,202508,1007,20,1,4409,5045,11060136,57633.49


In [7]:
# 성분(WK_COMPN_4), 요양기관종류(YOYANG_CLSFC_CD_ADJ), 입원·외래(OUT_IN_HOS) 기준 그룹화
summary_hos = df.groupby(['STD_YYYYMM','WK_COMPN_4', 'YOYANG_CLSFC_CD_ADJ', 'OUT_IN_HOS'], as_index=False).agg({
    'PATIENT_CNT': 'sum',        # 환자수
    'PRSCRPTN_TNDN_CNT': 'sum',  # 처방건수
    'PRSCRPTN_AMT': 'sum',       # 처방금액
    'PRSCRPTN_TOT_USE': 'sum'    # 총사용량
})

# 결과 확인
summary_hos.head()

,STD_YYYYMM,WK_COMPN_4,YOYANG_CLSFC_CD_ADJ,OUT_IN_HOS,PATIENT_CNT,PRSCRPTN_TNDN_CNT,PRSCRPTN_AMT,PRSCRPTN_TOT_USE
0,202508,1007,1,IN,8679,9543,55938749,329088.44
1,202508,1007,1,OUT,28525,32232,419692343,2122378.29
2,202508,1007,3,IN,251,295,834527,4511.00
3,202508,1007,3,OUT,126311,158361,465600446,2432864.89
4,202508,1007,7,IN,218,233,666446,3650.00
